# 📚 RAG Pipeline — Sistem Chatbot Edukasi Kesehatan Dasar Masyarakat
### Penerapan Retrieval-Augmented Generation (RAG) dengan Large Language Model (LLM)
**Judul Tugas:** PENERAPAN RETRIEVAL-AUGMENTED GENERATION (RAG) DENGAN LARGE LANGUAGE MODEL (LLM) UNTUK SISTEM CHATBOT EDUKASI KESEHATAN DASAR MASYARAKAT


# CELL 1 — Install Library RAG + Terjemahan  


In [1]:
!pip install pdfplumber pypdf -q
!pip install sentence-transformers -q
!pip install neo4j -q
!pip install rank-bm25 -q
!pip install scikit-learn nltk tqdm -q
!pip install groq openai -q

import nltk
nltk.download('stopwords', quiet=True)

print('✅ Semua library RAG + Terjemahan berhasil diinstall!')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 10.1 MB/s eta 0:00:00
✅ Semua library RAG + Terjemahan berhasil diinstall!


# CELL 2 — Konfigurasi


In [ ]:
import os

NEO4J_URI      = ''
NEO4J_USER     = ''
NEO4J_PASSWORD = ''

GROQ_API_KEY   = 'YOUR_GROQ_API_KEY'
LLM_PROVIDER   = 'groq'

LLM_MODELS = {
    'groq'  : 'llama-3.3-70b-versatile',
}
ACTIVE_MODEL = LLM_MODELS[LLM_PROVIDER]
EMBED_MODEL  = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

CHUNK_SIZE      = 300
CHUNK_OVERLAP   = 50
MIN_CHUNK_WORDS = 30
TOP_K           = 5
DOMAIN          = 'Kesehatan Masyarakat'

BOOK_METADATA = {
    'Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022': {
        'penulis'   : 'Atik Badiah',
        'tahun'     : 2022,
        'penerbit'  : 'Penerbit Ilmu Kesehatan',
        'topik_buku': 'Ilmu Kesehatan Masyarakat',
    },
    'Dasar-Dasar Ilmu Kesehatan Masyarakat': {
        'penulis'   : 'Tim Penulis',
        'tahun'     : 2020,
        'penerbit'  : 'Penerbit Kesehatan Indonesia',
        'topik_buku': 'Dasar Ilmu Kesehatan Masyarakat',
    },
}

CATEGORY = [
    'Epidemiologi dan Penyakit',
    'Promosi Kesehatan',
    'Gizi dan Kesehatan',
    'Kesehatan Lingkungan',
    'Kesehatan Ibu dan Anak',
    'Penyakit Menular',
    'Penyakit Tidak Menular',
    'Pelayanan Kesehatan',
    'Sanitasi dan Hygiene',
]

TRANSLATE_ENABLED    = False
USE_LLM_EVERY_N      = 3

os.environ['GROQ_API_KEY']   = GROQ_API_KEY

print('✅ Konfigurasi selesai!')
print(f'   LLM Provider : {LLM_PROVIDER.upper()} — {ACTIVE_MODEL}')
print(f'   Embed Model  : {EMBED_MODEL}')
print(f'   Chunk Size   : {CHUNK_SIZE} kata (overlap={CHUNK_OVERLAP})')
print(f'   Domain       : {DOMAIN}')
print()
print('📌 9 KATEGORI KESEHATAN MASYARAKAT:')
for i, c in enumerate(CATEGORY, 1):
    print(f'   {i}. {c}')
print()
print('📚 Sumber Buku:')
for title, meta in BOOK_METADATA.items():
    print(f'   • {title} ({meta["tahun"]})')


✅ Konfigurasi selesai!
   LLM Provider : GROQ — llama-3.3-70b-versatile
   Embed Model  : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
   Chunk Size   : 300 kata (overlap=50)
   Domain       : Kesehatan Masyarakat

📌 9 KATEGORI KESEHATAN MASYARAKAT:
   1. Epidemiologi dan Penyakit
   2. Promosi Kesehatan
   3. Gizi dan Kesehatan
   4. Kesehatan Lingkungan
   5. Kesehatan Ibu dan Anak
   6. Penyakit Menular
   7. Penyakit Tidak Menular
   8. Pelayanan Kesehatan
   9. Sanitasi dan Hygiene

📚 Sumber Buku:
   • Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022 (2022)
   • Dasar-Dasar Ilmu Kesehatan Masyarakat (2020)


# CELL 3 — Koneksi Neo4j


In [4]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def test_connection():
    with driver.session() as session:
        result = session.run("RETURN 'Koneksi berhasil ke Neo4j!' AS message")
        for record in result:
            print(record['message'])

test_connection()


Koneksi berhasil ke Neo4j!


# CELL 4 — Upload PDF Buku Kesehatan Masyarakat


In [5]:
from google.colab import files

print('📤 Upload file PDF buku kesehatan masyarakat (tersedia 2 buku):')
print('   1. BUKU DIGITAL ILMU KESEHATAN MASYARAKAT_ATIK BADIAH_TAHUN 2022.pdf')
print('   2. Dasar-Ilmu-Kesehatan-Masyarakat.pdf')
print('   (Upload satu per satu, atau upload keduanya sekaligus)')
uploaded_pdfs = files.upload()

if uploaded_pdfs:
    PDF_PATHS = []
    for filename in uploaded_pdfs.keys():
        path = f'/content/{filename}'
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'\n✅ Upload berhasil!')
        print(f'   File   : {filename}')
        print(f'   Path   : {path}')
        print(f'   Ukuran : {size_mb:.2f} MB')
        PDF_PATHS.append(path)
    print(f'\n📚 Total PDF siap diproses: {len(PDF_PATHS)} file')
else:
    print('❌ Tidak ada file yang diupload.')


📤 Upload file PDF buku kesehatan masyarakat (tersedia 2 buku):
   1. BUKU DIGITAL ILMU KESEHATAN MASYARAKAT_ATIK BADIAH_TAHUN 2022.pdf
   2. Dasar-Ilmu-Kesehatan-Masyarakat.pdf
   (Upload satu per satu, atau upload keduanya sekaligus)


Saving Dasar-Ilmu-Kesehatan-Masyarakat.pdf to Dasar-Ilmu-Kesehatan-Masyarakat.pdf
Saving BUKU DIGITAL ILMU KESEHATAN MASYARAKAT_ATIK BADIAH_TAHUN 2022.pdf to BUKU DIGITAL ILMU KESEHATAN MASYARAKAT_ATIK BADIAH_TAHUN 2022.pdf

✅ Upload berhasil!
   File   : Dasar-Ilmu-Kesehatan-Masyarakat.pdf
   Path   : /content/Dasar-Ilmu-Kesehatan-Masyarakat.pdf
   Ukuran : 3.86 MB

✅ Upload berhasil!
   File   : BUKU DIGITAL ILMU KESEHATAN MASYARAKAT_ATIK BADIAH_TAHUN 2022.pdf
   Path   : /content/BUKU DIGITAL ILMU KESEHATAN MASYARAKAT_ATIK BADIAH_TAHUN 2022.pdf
   Ukuran : 2.17 MB

📚 Total PDF siap diproses: 2 file


# CELL 5 — Data Ingestion (Ekstrak Teks dari PDF)


In [6]:
import pdfplumber, json, re
from tqdm import tqdm

def detect_chapter(text):
    patterns = [
        r'BAB\s+[IVXLCDM\d]+[\s:—–\-]+(.{3,60})',
        r'CHAPTER\s+\d+[\s:—–\-]+(.{3,60})',
        r'SECTION\s+\d+[\s:—–\-]+(.{3,60})',
        r'^(\d{1,2}\.\s+[A-Z][A-Za-z\s]{5,50})$',
    ]
    for pat in patterns:
        m = re.search(pat, text[:500], re.IGNORECASE | re.MULTILINE)
        if m:
            return m.group(1).strip() if m.lastindex else m.group(0).strip()
    return ''

def detect_source_label(pdf_path):
    fname = os.path.basename(pdf_path).replace('.pdf','').replace('_',' ').replace('-',' ')
    for key in BOOK_METADATA:
        key_words = key.lower().split()[:4]
        if any(w in fname.lower() for w in key_words):
            return key
    if 'atik' in fname.lower() or 'badiah' in fname.lower() or '2022' in fname:
        return 'Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022'
    elif 'dasar' in fname.lower():
        return 'Dasar-Dasar Ilmu Kesehatan Masyarakat'
    return fname[:60]

def ingest_pdf(pdf_path):
    source_label    = detect_source_label(pdf_path)
    book_meta       = BOOK_METADATA.get(source_label, {})
    pages           = []
    skipped         = 0
    current_chapter = 'Pendahuluan'

    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        print(f'   📚 Total halaman PDF : {total}')
        print(f'   📖 Sumber            : {source_label}')
        for i, page in enumerate(tqdm(pdf.pages, desc='   Ekstrak halaman', unit='hal')):
            text = page.extract_text()
            if not text or len(text.strip()) < 30:
                skipped += 1
                continue
            chapter = detect_chapter(text)
            if chapter:
                current_chapter = chapter
            pages.append({
                'content'         : text,
                'page'            : i + 1,
                'chapter'         : current_chapter,
                'source'          : source_label,
                'penulis'         : book_meta.get('penulis', 'Unknown'),
                'tahun'           : book_meta.get('tahun', 0),
                'penerbit'        : book_meta.get('penerbit', ''),
                'topik_buku'      : book_meta.get('topik_buku', DOMAIN),
                'category'        : CATEGORY,
                'domain'          : DOMAIN,
                'bahasa_asli'     : 'id',
                'word_count'      : len(text.split()),
            })
    return pages, skipped, total

print('=' * 60)
print('📥 DATA INGESTION — PDF Kesehatan Masyarakat → JSON')
print('=' * 60)

all_raw_pages = []
for pdf_path in PDF_PATHS:
    print(f'\n🔄 Memproses: {os.path.basename(pdf_path)}')
    raw_pages, skipped, total_pages = ingest_pdf(pdf_path)
    all_raw_pages.extend(raw_pages)
    print(f'   ✅ {len(raw_pages)}/{total_pages} halaman | Skip: {skipped}')

raw_pages = all_raw_pages
with open('/content/dataset_raw.json', 'w', encoding='utf-8') as f:
    json.dump(raw_pages, f, indent=2, ensure_ascii=False)

print(f'\n✅ Ingestion selesai! Total: {len(raw_pages)} halaman')
print(f'   Total kata: {sum(p["word_count"] for p in raw_pages):,}')


📥 DATA INGESTION — PDF Kesehatan Masyarakat → JSON

🔄 Memproses: Dasar-Ilmu-Kesehatan-Masyarakat.pdf
   📚 Total halaman PDF : 452
   📖 Sumber            : Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022


   Ekstrak halaman: 100%|██████████| 452/452 [00:35<00:00, 12.81hal/s]


   ✅ 441/452 halaman | Skip: 11

🔄 Memproses: BUKU DIGITAL ILMU KESEHATAN MASYARAKAT_ATIK BADIAH_TAHUN 2022.pdf
   📚 Total halaman PDF : 265
   📖 Sumber            : Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022


   Ekstrak halaman: 100%|██████████| 265/265 [00:26<00:00, 10.06hal/s]

   ✅ 255/265 halaman | Skip: 10

✅ Ingestion selesai! Total: 696 halaman
   Total kata: 102,906


# CELL 6 — Text Cleaning


In [7]:
import re, json

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'^\s*\d{1,4}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

with open('/content/dataset_raw.json', 'r', encoding='utf-8') as f:
    raw_pages = json.load(f)

cleaned_pages = []
short_drop    = 0

for page in raw_pages:
    c = clean_text(page['content'])
    page['content']    = c
    page['word_count'] = len(c.split())
    if page['word_count'] < 20:
        short_drop += 1
        continue
    cleaned_pages.append(page)

with open('/content/dataset_clean.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_pages, f, indent=2, ensure_ascii=False)

print(f'✅ Cleaning: {len(raw_pages)} → {len(cleaned_pages)} halaman (drop: {short_drop})')
print(f'   Total kata bersih : {sum(p["word_count"] for p in cleaned_pages):,}')


✅ Cleaning: 696 → 692 halaman (drop: 4)
   Total kata bersih : 102,106


# CELL 7 — Chunking (Sliding Window)


In [8]:
import json

def chunk_by_words(text, chunk_size, overlap):
    words = text.split()
    if len(words) < MIN_CHUNK_WORDS: return []
    if len(words) <= chunk_size:     return [text]
    chunks = []; start = 0
    while start < len(words):
        end   = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        if len(chunk.split()) >= MIN_CHUNK_WORDS:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

with open('/content/dataset_clean.json', 'r', encoding='utf-8') as f:
    cleaned_pages = json.load(f)

chunked_data = []
for page in cleaned_pages:
    chunks = chunk_by_words(page['content'], CHUNK_SIZE, CHUNK_OVERLAP)
    for i, chunk in enumerate(chunks):
        chunked_data.append({
            'chunk_id'      : f"page{page['page']}_chunk{i}",
            'content'       : chunk,
            'page'          : page['page'],
            'chapter'       : page.get('chapter', ''),
            'source'        : page['source'],
            'penulis'       : page.get('penulis', ''),
            'tahun'         : page.get('tahun', 0),
            'penerbit'      : page.get('penerbit', ''),
            'topik_buku'    : page.get('topik_buku', DOMAIN),
            'category'      : page['category'],
            'domain'        : page['domain'],
            'bahasa_asli'   : page.get('bahasa_asli', 'id'),
            'word_count'    : len(chunk.split()),
        })

with open('/content/dataset_chunked.json', 'w', encoding='utf-8') as f:
    json.dump(chunked_data, f, indent=2, ensure_ascii=False)

wc = [c['word_count'] for c in chunked_data]
print(f'✅ Chunking selesai! Total: {len(chunked_data)} chunks')
print(f'   Avg: {sum(wc)/len(wc):.0f} kata | Min: {min(wc)} | Max: {max(wc)}')


✅ Chunking selesai! Total: 689 chunks
   Avg: 148 kata | Min: 34 | Max: 300


# CELL 8 — Embedding (Sentence-Transformers, Multilingual)


In [9]:
import json
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print(f'📥 Loading embedding model: {EMBED_MODEL}')
embed_model = SentenceTransformer(EMBED_MODEL)
DIM = embed_model.get_sentence_embedding_dimension()
print(f'✅ Model loaded! Dimensi: {DIM}')

with open('/content/dataset_chunked.json', 'r', encoding='utf-8') as f:
    chunked_data = json.load(f)

BATCH_SIZE  = 64
texts       = [c['content'] for c in chunked_data]
all_vectors = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='   Batch embedding'):
    batch = texts[i : i + BATCH_SIZE]
    vecs  = embed_model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    all_vectors.extend(vecs.tolist())

for chunk, vec in zip(chunked_data, all_vectors):
    chunk['embedding'] = vec

with open('/content/dataset_embedded.json', 'w', encoding='utf-8') as f:
    json.dump(chunked_data, f, indent=2, ensure_ascii=False)

print(f'✅ Embedding selesai! {len(all_vectors)} vektor {DIM}-dim')


📥 Loading embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_13386/4032275542.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = embed_model.get_sentence_embedding_dimension()


✅ Model loaded! Dimensi: 384


   Batch embedding: 100%|██████████| 11/11 [01:22<00:00,  7.51s/it]


✅ Embedding selesai! 689 vektor 384-dim


# CELL 9 — Kamus Entitas Kesehatan Masyarakat


In [11]:
DISEASE_KW = [
    # Penyakit Menular
    'tuberkulosis', 'tb paru', 'tbc', 'tuberculosis',
    'malaria', 'demam berdarah', 'dbd', 'dengue hemorrhagic fever', 'dhf',
    'typhus', 'tifoid', 'demam tifoid', 'typhoid fever',
    'kolera', 'cholera', 'disentri', 'dysentery',
    'diare', 'gastroenteritis', 'diare akut',
    'pneumonia', 'ispa', 'infeksi saluran pernapasan', 'ari',
    'hiv', 'aids', 'hiv/aids',
    'hepatitis', 'hepatitis b', 'hepatitis c', 'hepatitis a',
    'cacar air', 'varicella', 'chickenpox',
    'campak', 'measles', 'morbili',
    'difteri', 'diphtheria', 'pertusis', 'whooping cough',
    'polio', 'poliomielitis',
    'tetanus', 'leptospirosis',
    'filariasis', 'kaki gajah', 'elephantiasis',
    'schistosomiasis', 'cacingan', 'helminthiasis',
    'rabies',
    # Penyakit Tidak Menular
    'diabetes mellitus', 'diabetes melitus', 'dm tipe 2', 'kencing manis',
    'hipertensi', 'tekanan darah tinggi', 'hypertension',
    'penyakit jantung', 'jantung koroner', 'coronary heart disease',
    'stroke', 'cerebrovascular accident',
    'kanker', 'cancer', 'tumor', 'neoplasma',
    'obesitas', 'kegemukan', 'overweight',
    'anemia', 'kurang darah',
    'malnutrisi', 'gizi buruk', 'kurang gizi', 'kwashiorkor', 'marasmus',
    'stunting', 'wasting', 'underweight',
    'penyakit paru obstruktif', 'ppok', 'copd',
]

SYMPTOM_KW = [
    'demam', 'fever', 'panas badan',
    'batuk', 'cough', 'batuk kronik', 'batuk berdahak',
    'sesak napas', 'dyspnea', 'kesulitan bernapas',
    'mual', 'nausea', 'muntah', 'vomiting',
    'diare', 'mencret', 'buang air besar cair',
    'nyeri perut', 'sakit perut', 'abdominal pain',
    'sakit kepala', 'headache', 'pusing',
    'lemas', 'fatigue', 'kelelahan',
    'penurunan berat badan', 'weight loss',
    'nafsu makan berkurang', 'anoreksia',
    'keringat malam', 'night sweats',
    'ruam kulit', 'rash', 'bintik merah',
    'ikterus', 'jaundice', 'kuning',
    'bengkak', 'edema', 'pembengkakan',
    'nyeri sendi', 'arthralgia',
    'pendarahan', 'bleeding', 'perdarahan',
]

PREVENTION_KW = [
    'vaksinasi', 'imunisasi', 'vaccination',
    'cuci tangan', 'hand washing', 'higiene tangan', 'ctps',
    'air bersih', 'clean water', 'air minum bersih',
    'sanitasi', 'sanitation', 'sanitasi lingkungan',
    'phbs', 'perilaku hidup bersih dan sehat',
    'jamban', 'toilet', 'wc', 'jamban sehat',
    'penggunaan kelambu', 'mosquito net', 'kelambu berinsektisida',
    '3m plus', 'pemberantasan sarang nyamuk', 'psn',
    'fogging', 'pengasapan',
    'pola makan sehat', 'gizi seimbang', 'makanan bergizi',
    'olahraga rutin', 'aktivitas fisik',
    'tidak merokok', 'stop rokok', 'anti rokok',
    'skrining', 'deteksi dini', 'early detection',
    'posyandu', 'puskesmas',
    'promosi kesehatan', 'penyuluhan kesehatan',
]

RISK_FACTOR_KW = [
    'kemiskinan', 'poverty', 'status ekonomi rendah',
    'kurang gizi', 'malnutrisi', 'gizi buruk',
    'lingkungan kumuh', 'permukiman padat',
    'air tidak bersih', 'air tercemar', 'sumber air tidak aman',
    'sanitasi buruk', 'tidak ada jamban',
    'merokok', 'smoking', 'kebiasaan merokok',
    'alkohol', 'minuman keras',
    'obesitas', 'overweight',
    'stres', 'stress',
    'kurang olahraga', 'sedentary lifestyle',
    'riwayat keluarga', 'faktor genetik',
    'paparan polusi', 'polusi udara', 'air pollution',
    'paparan pestisida', 'bahan kimia berbahaya',
    'kontak dengan pasien', 'transmisi langsung',
    'imunodefisiensi', 'imunitas lemah', 'daya tahan lemah',
    'tidak divaksin', 'belum diimunisasi',
    'perilaku berisiko', 'seks bebas',
    'penggunaan narkoba', 'napza', 'narkotika',
]

TREATMENT_KW = [
    'pengobatan', 'terapi', 'treatment', 'medikasi',
    'rawat inap', 'hospitalisasi', 'opname',
    'rawat jalan', 'poliklinik',
    'antibiotik', 'antibiotic',
    'antimalaria', 'antiviral', 'antifungal',
    'rehidrasi oral', 'oralit', 'cairan infus',
    'operasi', 'pembedahan', 'surgery',
    'kemoterapi', 'radiasi', 'radioterapi',
    'fisioterapi', 'rehabilitasi',
    'program dots', 'pengobatan tb',
    'art', 'antiretroviral', 'arv',
    'suplementasi gizi', 'pmpt', 'pmba',
    'konseling', 'psikoterapi',
]

HEALTH_SERVICE_KW = [
    'puskesmas', 'pusat kesehatan masyarakat',
    'rumah sakit', 'rs', 'hospital',
    'posyandu', 'pos pelayanan terpadu',
    'polindes', 'pondok bersalin desa',
    'bidan', 'dokter', 'perawat', 'tenaga kesehatan',
    'jkn', 'bpjs', 'jaminan kesehatan nasional',
    'asuransi kesehatan', 'health insurance',
    'pelayanan primer', 'primary health care',
    'pelayanan sekunder', 'pelayanan tersier',
    'posbindu', 'posyandu lansia',
    'p2p', 'p2ptm', 'program pemerintah',
    'dinkes', 'kemenkes', 'dinas kesehatan',
]

EPIDEMIOLOGY_KW = [
    'prevalensi', 'prevalence',
    'insidensi', 'incidence', 'insiden',
    'angka kematian', 'mortalitas', 'mortality', 'cfr',
    'angka kesakitan', 'morbiditas', 'morbidity',
    'distribusi penyakit', 'pola penyakit',
    'vektor', 'vector', 'reservoir',
    'transmisi', 'penularan', 'mode of transmission',
    'masa inkubasi', 'incubation period',
    'outbreak', 'kejadian luar biasa', 'klb', 'wabah', 'pandemi', 'epidemi',
    'surveilans', 'surveillance',
    'endemis', 'endemic', 'sporadic',
    'agent', 'host', 'environment', 'trias epidemiologi',
    'determinan kesehatan',
]

NUTRITION_KW = [
    'gizi', 'nutrition', 'nutrisi',
    'protein', 'karbohidrat', 'lemak', 'vitamin', 'mineral',
    'asi', 'air susu ibu', 'breastfeeding', 'menyusui',
    'mpasi', 'makanan pendamping asi',
    'stunting', 'wasting', 'gizi kurang', 'gizi buruk',
    'obesitas gizi lebih', 'overnutrition',
    'anemia gizi', 'kekurangan besi', 'defisiensi vitamin a',
    'tablet tambah darah', 'ttd', 'fe',
    'kms', 'kartu menuju sehat',
    'bb/u', 'tb/u', 'bb/tb',
    'gizi seimbang', 'pedoman gizi seimbang', 'pgs',
    '1000 hpk', 'seribu hari pertama kehidupan',
]

ENVIRONMENT_KW = [
    'kesehatan lingkungan', 'environmental health',
    'air bersih', 'sumber air', 'kualitas air',
    'limbah', 'sampah', 'waste management',
    'polusi udara', 'pencemaran udara', 'air pollution',
    'polusi air', 'pencemaran air',
    'sanitasi', 'hygiene',
    'jamban', 'jamban sehat', 'open defecation free', 'odf',
    'vektor penyakit', 'pengendalian vektor',
    'perumahan sehat', 'rumah sehat',
    'tempat pembuangan sampah', 'tps',
    'saluran pembuangan', 'drainase',
    'pestisida', 'insektisida', 'rodentisida',
]

MCH_KW = [
    'ibu hamil', 'kehamilan', 'antenatal care', 'anc', 'k1', 'k4',
    'persalinan', 'partus', 'delivery', 'melahirkan',
    'nifas', 'postpartum', 'ibu nifas',
    'bayi baru lahir', 'neonatus', 'newborn',
    'balita', 'bayi', 'anak bawah lima tahun',
    'kia', 'kesehatan ibu dan anak',
    'aki', 'angka kematian ibu', 'maternal mortality',
    'akb', 'angka kematian bayi', 'infant mortality',
    'akaba', 'angka kematian balita',
    'kb', 'keluarga berencana', 'family planning', 'kontrasepsi',
    'pemeriksaan kehamilan', 'buku kia',
    'imunisasi dasar', 'bcg', 'polio', 'dpt', 'campak',
]

PATIENT_GROUP_KW = [
    'bayi', 'infant', 'neonatus',
    'balita', 'anak bawah lima tahun',
    'anak usia sekolah', 'anak',
    'remaja', 'adolescent',
    'dewasa', 'adult', 'usia produktif',
    'lansia', 'lanjut usia', 'elderly',
    'ibu hamil', 'wanita hamil', 'pregnant women',
    'ibu menyusui',
    'masyarakat miskin', 'kelompok rentan',
    'komunitas', 'masyarakat',
    'pekerja', 'tenaga kerja', 'occupational',
]

HEALTH_PROMOTION_KW = [
    'promosi kesehatan', 'health promotion',
    'penyuluhan', 'edukasi kesehatan', 'health education',
    'komunikasi kesehatan', 'health communication',
    'perubahan perilaku', 'behavior change',
    'phbs', 'perilaku hidup bersih sehat',
    'pemberdayaan masyarakat', 'community empowerment',
    'advokasi', 'advocacy',
    'kader kesehatan', 'community health worker',
    'desa siaga', 'desa sehat',
    'germas', 'gerakan masyarakat hidup sehat',
    'media kesehatan', 'leaflet', 'poster', 'media komunikasi',
]

print('✅ Kamus entitas kesehatan masyarakat siap!')
print(f'   Disease keywords        : {len(DISEASE_KW)}')
print(f'   Symptom keywords        : {len(SYMPTOM_KW)}')
print(f'   Prevention keywords     : {len(PREVENTION_KW)}')
print(f'   Risk Factor keywords    : {len(RISK_FACTOR_KW)}')
print(f'   Treatment keywords      : {len(TREATMENT_KW)}')
print(f'   Health Service keywords : {len(HEALTH_SERVICE_KW)}')
print(f'   Epidemiology keywords   : {len(EPIDEMIOLOGY_KW)}')
print(f'   Nutrition keywords      : {len(NUTRITION_KW)}')
print(f'   Environment keywords    : {len(ENVIRONMENT_KW)}')
print(f'   MCH keywords            : {len(MCH_KW)}')
print(f'   Patient Group keywords  : {len(PATIENT_GROUP_KW)}')
print(f'   Health Promotion kw     : {len(HEALTH_PROMOTION_KW)}')


✅ Kamus entitas kesehatan masyarakat siap!
   Disease keywords        : 84
   Symptom keywords        : 46
   Prevention keywords     : 42
   Risk Factor keywords    : 43
   Treatment keywords      : 35
   Health Service keywords : 30
   Epidemiology keywords   : 38
   Nutrition keywords      : 36
   Environment keywords    : 30
   MCH keywords            : 40
   Patient Group keywords  : 26
   Health Promotion kw     : 25


# CELL 10 — Ekstraksi Entitas Klinis (Rule-based + LLM)


In [13]:
import json, re, time
from groq import Groq

llm_client = Groq(api_key=GROQ_API_KEY) if LLM_PROVIDER == 'groq' else None

def extract_rule_based(text: str) -> dict:
    t = text.lower()
    def match(kw_list):
        return list({kw for kw in kw_list if kw in t})
    return {
        'diseases'          : match(DISEASE_KW),
        'symptoms'          : match(SYMPTOM_KW),
        'preventions'       : match(PREVENTION_KW),
        'risk_factors'      : match(RISK_FACTOR_KW),
        'treatments'        : match(TREATMENT_KW),
        'health_services'   : match(HEALTH_SERVICE_KW),
        'epidemiology'      : match(EPIDEMIOLOGY_KW),
        'nutrition'         : match(NUTRITION_KW),
        'environment'       : match(ENVIRONMENT_KW),
        'mch'               : match(MCH_KW),
        'patient_groups'    : match(PATIENT_GROUP_KW),
        'health_promotion'  : match(HEALTH_PROMOTION_KW),
    }

def extract_llm(text: str) -> dict:
    prompt = f"""Dari teks kesehatan masyarakat berikut (Bahasa Indonesia), ekstrak entitas.
Kembalikan HANYA JSON valid (tanpa markdown, tanpa penjelasan) dengan format persis:
{{
  \"diseases\": [],
  \"symptoms\": [],
  \"preventions\": [],
  \"risk_factors\": [],
  \"treatments\": [],
  \"health_services\": [],
  \"epidemiology\": [],
  \"nutrition\": [],
  \"environment\": [],
  \"mch\": [],
  \"patient_groups\": [],
  \"health_promotion\": []
}}

Aturan: maks 5 item per kategori, gunakan Bahasa Indonesia, jangan duplikasi.

Teks: {text[:900]}"""
    try:
        resp = llm_client.chat.completions.create(
            model=ACTIVE_MODEL,
            messages=[{'role':'user','content':prompt}],
            max_tokens=500, temperature=0.0)
        raw = resp.choices[0].message.content
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            keys = ['diseases','symptoms','preventions','risk_factors','treatments',
                    'health_services','epidemiology','nutrition','environment',
                    'mch','patient_groups','health_promotion']
            return {k: parsed.get(k, []) for k in keys}
    except Exception:
        pass
    return extract_rule_based(text)

print('🔍 Mengekstrak entitas dari setiap chunk...')
from tqdm import tqdm
counters = {k: 0 for k in ['diseases','symptoms','preventions','risk_factors','treatments',
                            'health_services','epidemiology','nutrition','environment',
                            'mch','patient_groups','health_promotion']}

for i, chunk in enumerate(tqdm(chunked_data, desc='   Ekstrak entitas')):
    if i % USE_LLM_EVERY_N == 0:
        entities = extract_llm(chunk['content'])
        time.sleep(0.3)
    else:
        entities = extract_rule_based(chunk['content'])
    for k, v in entities.items():
        chunk[k] = v
        counters[k] += len(v)

print('\n✅ Ekstraksi selesai! Ringkasan entitas:')
for k, cnt in counters.items():
    bar = '█' * min(cnt // max(1, max(counters.values()) // 30), 30)
    print(f'   {k:<20} : {cnt:>5}  {bar}')

safe = [{k: v for k, v in c.items() if k != 'embedding'} for c in chunked_data]
with open('/content/dataset_chunked.json', 'w', encoding='utf-8') as f:
    json.dump(safe, f, indent=2, ensure_ascii=False)

🔍 Mengekstrak entitas dari setiap chunk...


   Ekstrak entitas: 100%|██████████| 689/689 [08:38<00:00,  1.33it/s]


✅ Ekstraksi selesai! Ringkasan entitas:
   diseases             :   832  ███████████████████████████
   symptoms             :    89  ██
   preventions          :   463  ███████████████
   risk_factors         :   263  ████████
   treatments           :   302  ██████████
   health_services      :   834  ███████████████████████████
   epidemiology         :   349  ███████████
   nutrition            :   902  ██████████████████████████████
   environment          :   276  █████████
   mch                  :   714  ███████████████████████
   patient_groups       :   915  ██████████████████████████████
   health_promotion     :   413  █████████████


# CELL 11 — Deteksi SubTopik Kesehatan Masyarakat per Chunk


In [14]:
from collections import Counter

def detect_subtopic(text):
    t = text.lower()

    # Penyakit Menular
    if any(k in t for k in ['tuberkulosis', 'tb paru', 'tbc', 'bta']):
        if any(k in t for k in ['dots', 'pengobatan tb', 'obat tb', 'art']):
            return 'TB - Pengobatan & DOTS'
        if any(k in t for k in ['penularan', 'transmisi', 'kontak']):
            return 'TB - Epidemiologi & Penularan'
        return 'TB - Umum'

    elif any(k in t for k in ['malaria', 'plasmodium', 'anopheles']):
        if any(k in t for k in ['kelambu', 'psn', 'pengendalian vektor']):
            return 'Malaria - Pencegahan & Pengendalian Vektor'
        if any(k in t for k in ['antimalaria', 'klorokuin', 'artemisinin']):
            return 'Malaria - Pengobatan'
        return 'Malaria - Umum'

    elif any(k in t for k in ['demam berdarah', 'dbd', 'dengue', 'aedes']):
        if any(k in t for k in ['3m', 'fogging', 'psn', 'pemberantasan sarang']):
            return 'DBD - Pengendalian Vektor'
        return 'DBD - Umum'

    elif any(k in t for k in ['diare', 'gastroenteritis', 'cholera', 'kolera']):
        if any(k in t for k in ['oralit', 'rehidrasi', 'zinc']):
            return 'Diare - Tatalaksana'
        if any(k in t for k in ['air bersih', 'sanitasi', 'cuci tangan']):
            return 'Diare - Pencegahan'
        return 'Diare - Umum'

    elif any(k in t for k in ['ispa', 'pneumonia', 'infeksi saluran pernapasan']):
        return 'ISPA & Pneumonia'

    elif any(k in t for k in ['hiv', 'aids', 'antiretroviral', 'arv']):
        return 'HIV/AIDS'

    # Gizi
    elif any(k in t for k in ['stunting', 'gizi buruk', 'gizi kurang', 'marasmus', 'kwashiorkor']):
        if any(k in t for k in ['1000 hpk', 'seribu hari', 'balita']):
            return 'Gizi - Stunting & 1000 HPK'
        return 'Gizi - Masalah Gizi'

    elif any(k in t for k in ['asi', 'air susu ibu', 'menyusui', 'mpasi']):
        return 'Gizi - ASI & MPASI'

    elif any(k in t for k in ['gizi seimbang', 'pedoman gizi', 'pgs', 'tumpeng gizi']):
        return 'Gizi - Gizi Seimbang'

    elif any(k in t for k in ['anemia', 'kekurangan besi', 'tablet fe', 'ttd']):
        return 'Gizi - Anemia & Defisiensi'

    # KIA
    elif any(k in t for k in ['ibu hamil', 'antenatal', 'anc', 'k1', 'k4']):
        return 'KIA - Antenatal Care'

    elif any(k in t for k in ['persalinan', 'partus', 'melahirkan', 'bersalin']):
        return 'KIA - Persalinan'

    elif any(k in t for k in ['aki', 'angka kematian ibu', 'maternal mortality']):
        return 'KIA - Kematian Ibu'

    elif any(k in t for k in ['akb', 'angka kematian bayi', 'neonatus', 'bayi baru lahir']):
        return 'KIA - Kematian Bayi & Neonatal'

    elif any(k in t for k in ['kb', 'keluarga berencana', 'kontrasepsi', 'family planning']):
        return 'KIA - Keluarga Berencana'

    # Imunisasi
    elif any(k in t for k in ['imunisasi', 'vaksinasi', 'vaksin', 'posyandu']):
        if any(k in t for k in ['campak', 'polio', 'bcg', 'dpt', 'hepatitis b', 'hb']):
            return 'Imunisasi - Imunisasi Dasar'
        return 'Imunisasi - Program Imunisasi'

    # Kesehatan Lingkungan
    elif any(k in t for k in ['sanitasi', 'jamban', 'odf', 'open defecation']):
        return 'Kesling - Sanitasi & Jamban'

    elif any(k in t for k in ['air bersih', 'kualitas air', 'sumber air', 'air minum']):
        return 'Kesling - Air Bersih'

    elif any(k in t for k in ['sampah', 'limbah', 'pengelolaan sampah', 'tps']):
        return 'Kesling - Pengelolaan Sampah'

    elif any(k in t for k in ['polusi udara', 'pencemaran', 'rokok', 'merokok']):
        return 'Kesling - Polusi & Rokok'

    # PTM
    elif any(k in t for k in ['diabetes', 'dm', 'kencing manis', 'gula darah']):
        return 'PTM - Diabetes Mellitus'

    elif any(k in t for k in ['hipertensi', 'tekanan darah tinggi', 'darah tinggi']):
        return 'PTM - Hipertensi'

    elif any(k in t for k in ['jantung', 'stroke', 'kardiovaskular', 'cvd']):
        return 'PTM - Penyakit Kardiovaskular'

    elif any(k in t for k in ['kanker', 'cancer', 'tumor', 'neoplasma']):
        return 'PTM - Kanker'

    elif any(k in t for k in ['obesitas', 'overweight', 'berat badan lebih']):
        return 'PTM - Obesitas'

    # Promosi Kesehatan
    elif any(k in t for k in ['promosi kesehatan', 'penyuluhan', 'edukasi kesehatan']):
        return 'Promkes - Penyuluhan & Edukasi'

    elif any(k in t for k in ['phbs', 'perilaku hidup bersih', 'germas']):
        return 'Promkes - PHBS & GERMAS'

    elif any(k in t for k in ['pemberdayaan masyarakat', 'kader', 'desa siaga']):
        return 'Promkes - Pemberdayaan Masyarakat'

    # Pelayanan Kesehatan
    elif any(k in t for k in ['puskesmas', 'pusat kesehatan masyarakat']):
        return 'Yankes - Puskesmas'

    elif any(k in t for k in ['jkn', 'bpjs', 'jaminan kesehatan', 'asuransi']):
        return 'Yankes - JKN & BPJS'

    # Epidemiologi
    elif any(k in t for k in ['epidemiologi', 'surveilans', 'surveillance', 'klb', 'outbreak']):
        return 'Epidemiologi - Surveilans & KLB'

    elif any(k in t for k in ['prevalensi', 'insidensi', 'mortalitas', 'morbiditas']):
        return 'Epidemiologi - Statistik Kesehatan'

    return 'Umum - Kesehatan Masyarakat'


for item in chunked_data:
    item['subtopic'] = detect_subtopic(item.get('content', ''))

dist = Counter(item['subtopic'] for item in chunked_data)
print(f'✅ SubTopik: {len(dist)} unik dari {len(chunked_data)} chunk')
print('\n📊 Top-20 SubTopik Kesehatan Masyarakat:')
for st, cnt in dist.most_common(20):
    bar = '█' * min(cnt, 35)
    print(f'   {st:<52} {cnt:>4}x  {bar}')


✅ SubTopik: 24 unik dari 689 chunk

📊 Top-20 SubTopik Kesehatan Masyarakat:
   Gizi - ASI & MPASI                                    484x  ███████████████████████████████████
   KIA - Kematian Ibu                                     57x  ███████████████████████████████████
   Gizi - Stunting & 1000 HPK                             22x  ██████████████████████
   HIV/AIDS                                               19x  ███████████████████
   Malaria - Umum                                         16x  ████████████████
   Gizi - Masalah Gizi                                    14x  ██████████████
   Diare - Umum                                           11x  ███████████
   Umum - Kesehatan Masyarakat                            10x  ██████████
   TB - Umum                                              10x  ██████████
   KIA - Antenatal Care                                    8x  ████████
   DBD - Umum                                              7x  ███████
   Promkes - Penyuluhan & Edukasi

# CELL 12 — Setup Schema Neo4j (Kesehatan Masyarakat)


In [15]:
def setup_schema(driver):
    with driver.session() as s:
        s.run('MATCH (n) DETACH DELETE n')
        constraints = [
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Chunk)          REQUIRE n.chunk_id IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Source)         REQUIRE n.title    IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Buku)           REQUIRE n.judul    IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Penulis)        REQUIRE n.nama     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Page)           REQUIRE n.number   IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Disease)        REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Symptom)        REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Prevention)     REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:RiskFactor)     REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Treatment)      REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:HealthService)  REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Epidemiology)   REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Nutrition)      REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Environment)    REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:MCH)            REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:PatientGroup)   REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:HealthPromotion) REQUIRE n.name    IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:Category)       REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:SubTopik)       REQUIRE n.name     IS UNIQUE',
            'CREATE CONSTRAINT IF NOT EXISTS FOR (n:TopikBuku)      REQUIRE n.name     IS UNIQUE',
        ]
        for q in constraints:
            try: s.run(q)
            except Exception as e: print(f'   ⚠️  {e}')
    print('✅ Schema Neo4j siap — domain Kesehatan Masyarakat')

setup_schema(driver)


✅ Schema Neo4j siap — domain Kesehatan Masyarakat


# CELL 13A — Import Node Buku, Penulis, TopikBuku


In [16]:
def import_book_nodes(driver):
    with driver.session() as session:
        for source_title, meta in BOOK_METADATA.items():
            session.run("""
                MERGE (src:Source {title: $src_title})
                MERGE (b:Buku {judul: $src_title})
                SET   b.penerbit = $penerbit, b.tahun = $tahun, b.domain = $domain
                MERGE (src)-[:MERUPAKAN_BUKU]->(b)
                MERGE (pn:Penulis {nama: $penulis})
                MERGE (b)-[:DITULIS_OLEH]->(pn)
                MERGE (tb:TopikBuku {name: $topik})
                MERGE (b)-[:BERKAITAN_DENGAN]->(tb)
            """, {
                'src_title': source_title,
                'penerbit' : meta['penerbit'],
                'tahun'    : meta['tahun'],
                'penulis'  : meta['penulis'],
                'topik'    : meta['topik_buku'],
                'domain'   : DOMAIN,
            })
    print('✅ Node Buku, Penulis, TopikBuku di-import!')
    for st, m in BOOK_METADATA.items():
        print(f'   📖 {st[:60]}')
        print(f'      Penulis : {m["penulis"]}')

import_book_nodes(driver)


✅ Node Buku, Penulis, TopikBuku di-import!
   📖 Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022
      Penulis : Atik Badiah
   📖 Dasar-Dasar Ilmu Kesehatan Masyarakat
      Penulis : Tim Penulis


# CELL 13B — Import Chunk + Core Relasi


In [17]:
def import_chunk_core(tx, batch):
    tx.run("""
        UNWIND $batch AS row
        MERGE (c:Chunk {chunk_id: row.chunk_id})
        SET   c.content = row.content, c.page = row.page, c.chapter = row.chapter,
              c.source = row.source, c.domain = row.domain, c.word_count = row.word_count,
              c.embedding = row.embedding, c.subtopic = row.subtopic
        MERGE (p:Page {number: row.page})
        MERGE (c)-[:BERADA_DI]->(p)
        MERGE (src:Source {title: row.source})
        MERGE (c)-[:DIAMBIL_DARI]->(src)
        MERGE (st:SubTopik {name: row.subtopic})
        MERGE (c)-[:MEMBAHAS_TOPIK]->(st)
        WITH c, st, row
        UNWIND row.categories AS catName
        MERGE (cat:Category {name: catName})
        MERGE (st)-[:BAGIAN_DARI]->(cat)
    """, {'batch': batch})

BATCH_SIZE = 100
print(f'🔄 Import {len(chunked_data)} chunk ke Neo4j...')

with driver.session() as session:
    for i in range(0, len(chunked_data), BATCH_SIZE):
        batch_raw = chunked_data[i : i + BATCH_SIZE]
        batch = [{
            'chunk_id'  : c['chunk_id'],
            'content'   : c['content'],
            'page'      : c['page'],
            'chapter'   : c.get('chapter', ''),
            'source'    : c['source'],
            'domain'    : c['domain'],
            'word_count': c['word_count'],
            'embedding' : c.get('embedding', []),
            'subtopic'  : c.get('subtopic', 'Umum - Kesehatan Masyarakat'),
            'categories': c.get('category', CATEGORY) if isinstance(c.get('category'), list) else [c.get('category', 'Umum')],
        } for c in batch_raw]
        session.execute_write(import_chunk_core, batch)
        print(f'   ✅ Batch {i}–{i+len(batch)}')

print('\n🔥 Import chunk selesai!')


🔄 Import 689 chunk ke Neo4j...
   ✅ Batch 0–100
   ✅ Batch 100–200
   ✅ Batch 200–300
   ✅ Batch 300–400
   ✅ Batch 400–500
   ✅ Batch 500–600
   ✅ Batch 600–689

🔥 Import chunk selesai!


# CELL 13C — Import Entitas + Relasi Chunk→Entitas


In [18]:
ENTITY_MAP = [
    ('diseases',         'Disease',       'MENYEBUT_PENYAKIT'),
    ('symptoms',         'Symptom',       'MENYEBUT_GEJALA'),
    ('preventions',      'Prevention',    'MENYEBUT_PENCEGAHAN'),
    ('risk_factors',     'RiskFactor',    'MENYEBUT_FAKTOR_RISIKO'),
    ('treatments',       'Treatment',     'MENYEBUT_TERAPI'),
    ('health_services',  'HealthService', 'MENYEBUT_YANKES'),
    ('epidemiology',     'Epidemiology',  'MENYEBUT_EPIDEMIOLOGI'),
    ('nutrition',        'Nutrition',     'MENYEBUT_GIZI'),
    ('environment',      'Environment',   'MENYEBUT_LINGKUNGAN'),
    ('mch',              'MCH',           'MENYEBUT_KIA'),
    ('patient_groups',   'PatientGroup',  'MENYEBUT_KELOMPOK_SASARAN'),
    ('health_promotion', 'HealthPromotion','MENYEBUT_PROMKES'),
]

def import_entities_batch(tx, chunk_id, field, label, rel_type, values):
    if not values: return
    cypher = f"""
        UNWIND $values AS val
        MERGE (e:{label} {{name: val}})
        WITH e
        MATCH (c:Chunk {{chunk_id: $cid}})
        MERGE (c)-[:{rel_type}]->(e)
    """
    tx.run(cypher, {'values': [v.title() for v in values], 'cid': chunk_id})

print('🔗 Import entitas kesehatan masyarakat ke Neo4j...')
from tqdm import tqdm

with driver.session() as session:
    for chunk in tqdm(chunked_data, desc='   Import entitas'):
        cid = chunk['chunk_id']
        for field, label, rel_type in ENTITY_MAP:
            values = chunk.get(field, [])
            if values:
                session.execute_write(import_entities_batch, cid, field, label, rel_type, values)

print('\n✅ Semua entitas selesai di-import!')


🔗 Import entitas kesehatan masyarakat ke Neo4j...


   Import entitas: 100%|██████████| 689/689 [42:30<00:00,  3.70s/it]


✅ Semua entitas selesai di-import!


# CELL 13D — Bangun Relasi Antar-Entitas (Knowledge Inference)


In [19]:
def build_inference_relations(driver):
    queries = [
        ('Disease → Symptom',
         'MATCH (c:Chunk)-[:MENYEBUT_PENYAKIT]->(d:Disease), (c)-[:MENYEBUT_GEJALA]->(s:Symptom) MERGE (d)-[:MEMILIKI_GEJALA]->(s)'),
        ('Disease → Prevention',
         'MATCH (c:Chunk)-[:MENYEBUT_PENYAKIT]->(d:Disease), (c)-[:MENYEBUT_PENCEGAHAN]->(prev:Prevention) MERGE (d)-[:DICEGAH_DENGAN]->(prev)'),
        ('Disease → RiskFactor',
         'MATCH (c:Chunk)-[:MENYEBUT_PENYAKIT]->(d:Disease), (c)-[:MENYEBUT_FAKTOR_RISIKO]->(rf:RiskFactor) MERGE (d)-[:DIPICU_OLEH]->(rf)'),
        ('Disease → Treatment',
         'MATCH (c:Chunk)-[:MENYEBUT_PENYAKIT]->(d:Disease), (c)-[:MENYEBUT_TERAPI]->(t:Treatment) MERGE (d)-[:DITANGANI_DENGAN]->(t)'),
        ('Disease → PatientGroup',
         'MATCH (c:Chunk)-[:MENYEBUT_PENYAKIT]->(d:Disease), (c)-[:MENYEBUT_KELOMPOK_SASARAN]->(pg:PatientGroup) MERGE (d)-[:UMUM_PADA]->(pg)'),
        ('Disease → HealthService',
         'MATCH (c:Chunk)-[:MENYEBUT_PENYAKIT]->(d:Disease), (c)-[:MENYEBUT_YANKES]->(hs:HealthService) MERGE (d)-[:DITANGANI_DI]->(hs)'),
        ('Disease → Epidemiology',
         'MATCH (c:Chunk)-[:MENYEBUT_PENYAKIT]->(d:Disease), (c)-[:MENYEBUT_EPIDEMIOLOGI]->(ep:Epidemiology) MERGE (d)-[:MEMILIKI_DATA_EPIDEMIOLOGI]->(ep)'),
        ('Disease → Environment',
         'MATCH (c:Chunk)-[:MENYEBUT_PENYAKIT]->(d:Disease), (c)-[:MENYEBUT_LINGKUNGAN]->(env:Environment) MERGE (d)-[:TERKAIT_LINGKUNGAN]->(env)'),
        ('Nutrition → PatientGroup',
         'MATCH (c:Chunk)-[:MENYEBUT_GIZI]->(n:Nutrition), (c)-[:MENYEBUT_KELOMPOK_SASARAN]->(pg:PatientGroup) MERGE (n)-[:DIPERUNTUKKAN_UNTUK]->(pg)'),
        ('Prevention → Disease',
         'MATCH (c:Chunk)-[:MENYEBUT_PENCEGAHAN]->(prev:Prevention), (c)-[:MENYEBUT_PENYAKIT]->(d:Disease) MERGE (prev)-[:MENCEGAH]->(d)'),
        ('HealthPromotion → Disease',
         'MATCH (c:Chunk)-[:MENYEBUT_PROMKES]->(hp:HealthPromotion), (c)-[:MENYEBUT_PENYAKIT]->(d:Disease) MERGE (hp)-[:BERTUJUAN_MENCEGAH]->(d)'),
        ('MCH → PatientGroup',
         'MATCH (c:Chunk)-[:MENYEBUT_KIA]->(m:MCH), (c)-[:MENYEBUT_KELOMPOK_SASARAN]->(pg:PatientGroup) MERGE (m)-[:MELAYANI]->(pg)'),
    ]
    with driver.session() as s:
        for label, cypher in queries:
            try:
                result = s.run(cypher)
                rels   = result.consume().counters.relationships_created
                print(f'   ✅ {label:<45} +{rels} relasi')
            except Exception as e:
                print(f'   ❌ {label}: {e}')

print('🔗 Membangun relasi antar-entitas kesehatan masyarakat...')
build_inference_relations(driver)
print('\n🔥 Semua relasi selesai dibangun!')


🔗 Membangun relasi antar-entitas kesehatan masyarakat...
   ✅ Disease → Symptom                             +234 relasi
   ✅ Disease → Prevention                          +549 relasi
   ✅ Disease → RiskFactor                          +568 relasi
   ✅ Disease → Treatment                           +320 relasi
   ✅ Disease → PatientGroup                        +551 relasi
   ✅ Disease → HealthService                       +486 relasi
   ✅ Disease → Epidemiology                        +545 relasi
   ✅ Disease → Environment                         +381 relasi
   ✅ Nutrition → PatientGroup                      +314 relasi
   ✅ Prevention → Disease                          +549 relasi
   ✅ HealthPromotion → Disease                     +350 relasi
   ✅ MCH → PatientGroup                            +391 relasi

🔥 Semua relasi selesai dibangun!


# CELL 14 — Simpan Dataset Chunked (tanpa embedding)


In [20]:
import json

chunked_no_vec = [{k: v for k, v in c.items() if k != 'embedding'} for c in chunked_data]

with open('/content/dataset_chunked.json', 'w', encoding='utf-8') as f:
    json.dump(chunked_no_vec, f, indent=2, ensure_ascii=False)

print(f'✅ Dataset chunked (tanpa embedding) disimpan: {len(chunked_no_vec)} chunk')
print('   Preview chunk pertama (field lengkap):')
for k, v in list(chunked_no_vec[0].items()):
    if k not in ('content',):
        print(f'   {k:<20}: {str(v)[:80]}')


✅ Dataset chunked (tanpa embedding) disimpan: 689 chunk
   Preview chunk pertama (field lengkap):
   chunk_id            : page1_chunk0
   page                : 1
   chapter             : Pendahuluan
   source              : Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022
   penulis             : Atik Badiah
   tahun               : 2022
   penerbit            : Penerbit Ilmu Kesehatan
   topik_buku          : Ilmu Kesehatan Masyarakat
   category            : ['Epidemiologi dan Penyakit', 'Promosi Kesehatan', 'Gizi dan Kesehatan', 'Keseha
   domain              : Kesehatan Masyarakat
   bahasa_asli         : id
   word_count          : 103
   diseases            : ['DBD']
   symptoms            : []
   preventions         : []
   risk_factors        : []
   treatments          : []
   health_services     : ['Institut Kesehatan Prima Nusantara']
   epidemiology        : ['PATH ANALYSIS OF INCIDENT']
   nutrition           : []
   environment         : ['Kota Bukittinggi']
   

# CELL 15 — Verifikasi Knowledge Graph Neo4j


In [21]:
def verify_graph(driver):
    with driver.session() as s:
        node_labels = [
            ('Chunk','📄'),('Page','📑'),('Source','📚'),('Buku','📖'),
            ('Penulis','✍️'),('TopikBuku','🏷️'),('Category','🗂️'),('SubTopik','🔖'),
            ('Disease','🦠'),('Symptom','🩺'),('Prevention','🛡️'),('RiskFactor','⚡'),
            ('Treatment','💉'),('HealthService','🏥'),('Epidemiology','📊'),
            ('Nutrition','🥗'),('Environment','🌿'),('MCH','👶'),
            ('PatientGroup','👥'),('HealthPromotion','📣'),
        ]
        node_counts = {}
        for label, _ in node_labels:
            node_counts[label] = s.run(f'MATCH (n:{label}) RETURN count(n) AS c').single()['c']

    SEP = '=' * 65
    print(SEP)
    print('    RINGKASAN KNOWLEDGE GRAPH — KESEHATAN MASYARAKAT')
    print(SEP)
    print('\n📦 NODE:')
    total_nodes = sum(node_counts.values())
    emoji_map = {label: icon for label, icon in node_labels}
    for label, cnt in node_counts.items():
        bar = '█' * min(cnt, 30)
        print(f'   {emoji_map.get(label,"")} {label:<20} {cnt:>5}  {bar}')
    print(f'\n   TOTAL NODE: {total_nodes:,}')
    print(SEP)
    print('✅ Knowledge Graph Kesehatan Masyarakat siap!')
    print('   Lanjutkan ke LLM_KesehatanMasyarakat.ipynb')

verify_graph(driver)


    RINGKASAN KNOWLEDGE GRAPH — KESEHATAN MASYARAKAT

📦 NODE:
   📄 Chunk                  446  ██████████████████████████████
   📑 Page                   445  ██████████████████████████████
   📚 Source                   2  ██
   📖 Buku                     2  ██
   ✍️ Penulis                  2  ██
   🏷️ TopikBuku                2  ██
   🗂️ Category                 9  █████████
   🔖 SubTopik                24  ████████████████████████
   🦠 Disease                114  ██████████████████████████████
   🩺 Symptom                 57  ██████████████████████████████
   🛡️ Prevention             144  ██████████████████████████████
   ⚡ RiskFactor             144  ██████████████████████████████
   💉 Treatment               39  ██████████████████████████████
   🏥 HealthService          104  ██████████████████████████████
   📊 Epidemiology            93  ██████████████████████████████
   🥗 Nutrition               54  ██████████████████████████████
   🌿 Environment             81  ████████████████

# CELL 16 — Export & Download Dataset ke Google Drive


In [22]:
import shutil, os, json
from datetime import datetime
from google.colab import files as colab_files

ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_DIR = '/content/drive/MyDrive/kesmas_rag_output'
os.makedirs(SAVE_DIR, exist_ok=True)

files_map = {
    '/content/dataset_raw.json'     : f'dataset_raw_{ts}.json',
    '/content/dataset_clean.json'   : f'dataset_clean_{ts}.json',
    '/content/dataset_chunked.json' : f'dataset_chunked_{ts}.json',
    '/content/dataset_embedded.json': f'dataset_embedded_{ts}.json',
}

print('💾 Menyimpan ke Google Drive...')
for src, dst_name in files_map.items():
    dst = os.path.join(SAVE_DIR, dst_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        size = os.path.getsize(dst) / 1024
        print(f'   ✅ {dst_name:<55} {size:>7.1f} KB')
    else:
        print(f'   ⚠️  {src} tidak ditemukan — skip')

print(f'\n📁 Tersimpan di: {SAVE_DIR}')
print('\n📥 Download dataset_embedded.json (dibutuhkan LLM_KesehatanMasyarakat.ipynb)...')
colab_files.download('/content/dataset_embedded.json')
print('✅ Selesai!')


💾 Menyimpan ke Google Drive...
   ✅ dataset_raw_20260429_160453.json                         1266.1 KB
   ✅ dataset_clean_20260429_160453.json                       1237.8 KB
   ✅ dataset_chunked_20260429_160453.json                     1600.3 KB
   ✅ dataset_embedded_20260429_160453.json                    8494.7 KB

📁 Tersimpan di: /content/drive/MyDrive/kesmas_rag_output

📥 Download dataset_embedded.json (dibutuhkan LLM_KesehatanMasyarakat.ipynb)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Selesai!
